In [0]:
# Legacy hardcoded Kafka credentials (deprecated - now using secrets)
# These credentials are no longer used and kept for reference only
# bootstrap_server='pkc-xrnwx.asia-south2.gcp.confluent.cloud:9092'
# api_key='G3JBBMJJCOVKMBTE'
# api_secret='cfltaJ/I+NTkTBUXyNTbt/X+aS7gQYMfIM7+IPT9NNI6vOr3U5zR4edO1deplEmg'
# topic_name='credit_card_transactions'

In [0]:
# Retrieve Kafka connection details securely from Databricks secrets
# Secrets scope: fraudradar-scope contains encrypted connection credentials
import json

kafka_connection_json = dbutils.secrets.get(scope="fraudradar-scope",key="kafka_connection_details")
kafka_config = json.loads(kafka_connection_json)
bootstrap_server=kafka_config['bootstrap_servers']
api_key=kafka_config['api_key']
api_secret=kafka_config['api_secret']
topic_name=kafka_config['topic']

In [0]:
# Build JAAS configuration string for Kafka SASL authentication
# Uses PLAIN mechanism with API key and secret for Confluent Cloud
jaas_config=f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'

In [0]:
# Read Kafka data in batch mode from earliest offset
# This is useful for testing connectivity and inspecting existing messages
sample_batch_data = (spark.read.format("kafka")
                     .option("kafka.bootstrap.servers", bootstrap_server)
                     .option("subscribe", topic_name)
                     .option("kafka.security.protocol", "SASL_SSL")
                     .option("kafka.sasl.mechanism", "PLAIN")
                     .option("kafka.sasl.jaas.config", jaas_config)
                     .option("startingOffsets","earliest")
                     .load())

In [0]:
# Count total messages available in the Kafka topic
sample_batch_data.count()

In [0]:
# Display raw Kafka messages with metadata (key, value as binary, partition, offset, timestamp)
display(sample_batch_data)

In [0]:
# Parse and cast Kafka message fields to readable string format
# Converts binary key/value columns to strings for easier inspection
from pyspark.sql.functions import col

parsed_batch_data = sample_batch_data.select(
    col("key").cast("string"),
    col("value").cast("string"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")
)

In [0]:
# Display parsed Kafka messages with readable string values
display(parsed_batch_data)

In [0]:
# Save batch test data to bronze layer table
# Mode: overwrite - replaces existing data for testing purposes
parsed_batch_data.write.mode("overwrite").saveAsTable("fraudradar.bronze.transactions_batch_test")

In [0]:
# Create streaming DataFrame to continuously read from Kafka
# Uses readStream for real-time data ingestion starting from earliest offset
sample_streaming_data = (spark.readStream.format("kafka")
                     .option("kafka.bootstrap.servers", bootstrap_server)
                     .option("subscribe", topic_name)
                     .option("kafka.security.protocol", "SASL_SSL")
                     .option("kafka.sasl.mechanism", "PLAIN")
                     .option("kafka.sasl.jaas.config", jaas_config)
                     .option("startingOffsets","earliest")
                     .load())

In [0]:
# Parse streaming Kafka messages - cast binary fields to strings
# Same transformation as batch mode but applied to streaming data
from pyspark.sql.functions import col

parsed_streaming_data = sample_streaming_data.select(
    col("key").cast("string"),
    col("value").cast("string"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("timestampType")
)

In [0]:
# Write streaming data to Delta table in bronze layer
# - Checkpoint location: tracks processed offsets for fault tolerance
# - Trigger: availableNow processes all available data then stops (micro-batch)
# - OutputMode: append for inserting new records only
streaming_query = (parsed_streaming_data.writeStream.format("delta")
.outputMode("append")
.option("checkpointLocation", "/Volumes/fraudradar/source/checkpoint")
.trigger(availableNow=True)
.toTable("fraudradar.bronze.transactions_streaming_test"))

print("Query Id : ",streaming_query.id)

In [0]:
%sql
-- Query the streaming test table to verify data ingestion
-- Shows all records written by the streaming query
SELECT * FROM fraudradar.bronze.transactions_streaming_test